In [13]:
import xgboost as xgb
print(xgb.__version__)

3.0.4


In [14]:
import sys, xgboost as xgb
print(sys.executable)        # should point to .../.venv/bin/python
print(xgb.__version__)       # should print 3.0.4
print(xgb.__file__)          # should live under .../.venv/...

e:\Regression_ML_EndtoEnd\.venv\Scripts\python.exe
3.0.4
e:\Regression_ML_EndtoEnd\.venv\Lib\site-packages\xgboost\__init__.py


In [15]:
# ==============================================
# 1. Imports
# ==============================================
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor
import optuna
import mlflow
import mlflow.xgboost

In [16]:
# ==============================================
# 2. Load processed datasets
# ==============================================
train_df = pd.read_csv(r"E:\Regression_ML_EndtoEnd\data\processed\feature_engineered_train.csv")
eval_df  = pd.read_csv(r"E:\Regression_ML_EndtoEnd\data\processed\feature_engineered_eval.csv")


# Define target + features
target = "price"
X_train, y_train = train_df.drop(columns=[target]), train_df[target]
X_eval, y_eval   = eval_df.drop(columns=[target]), eval_df[target]

print("Train shape:", X_train.shape)
print("Eval shape:", X_eval.shape)

Train shape: (576815, 39)
Eval shape: (148448, 39)


In [17]:
# ==============================================
# 3. Define Optuna objective function with MLflow
# ==============================================
def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 200, 1000),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "gamma": trial.suggest_float("gamma", 0.0, 5.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        "random_state": 42,
        "n_jobs": -1,
        "tree_method": "hist",
    }

    with mlflow.start_run(nested=True):
        model = XGBRegressor(**params)
        model.fit(X_train, y_train)

        y_pred = model.predict(X_eval)
        rmse = float(np.sqrt(mean_squared_error(y_eval, y_pred)))
        mae = float(mean_absolute_error(y_eval, y_pred))
        r2 = float(r2_score(y_eval, y_pred))

        # Log hyperparameters + metrics
        mlflow.log_params(params)
        mlflow.log_metrics({"rmse": rmse, "mae": mae, "r2": r2})

    return rmse

In [18]:
# ==============================================
# 4. Run Optuna study with MLflow
# ==============================================
# Force MLflow to always use the root project mlruns folder
mlflow.set_tracking_uri("file:///E:/Regression_ML_EndtoEnd/mlruns")
mlflow.set_experiment("xgboost_optuna_housing")

study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=15)

print("Best params:", study.best_trial.params)

[I 2026-03-24 22:52:14,938] A new study created in memory with name: no-name-b9df705b-38e0-40d3-83fb-86d262354327
[I 2026-03-24 22:52:29,078] Trial 0 finished with value: 71974.18317115371 and parameters: {'n_estimators': 615, 'max_depth': 8, 'learning_rate': 0.03145780675307168, 'subsample': 0.8212543912129113, 'colsample_bytree': 0.9395966783289462, 'min_child_weight': 1, 'gamma': 4.851605734802905, 'reg_alpha': 1.1073146779563353e-05, 'reg_lambda': 9.424756786434244}. Best is trial 0 with value: 71974.18317115371.
[I 2026-03-24 22:52:43,163] Trial 1 finished with value: 73173.7175465533 and parameters: {'n_estimators': 599, 'max_depth': 8, 'learning_rate': 0.05744779311856923, 'subsample': 0.6274710232655938, 'colsample_bytree': 0.7302727077308191, 'min_child_weight': 6, 'gamma': 3.6459772648554494, 'reg_alpha': 0.00904238039879468, 'reg_lambda': 0.00022022997953192075}. Best is trial 0 with value: 71974.18317115371.
[I 2026-03-24 22:52:49,940] Trial 2 finished with value: 77417.550

Best params: {'n_estimators': 571, 'max_depth': 6, 'learning_rate': 0.03917308213845372, 'subsample': 0.5877863037262802, 'colsample_bytree': 0.5706416391711551, 'min_child_weight': 3, 'gamma': 3.5553873000370704, 'reg_alpha': 0.0001276641990790014, 'reg_lambda': 0.7992779148608345}


In [19]:
# ==============================================
# 5. Train final model with best params and log to MLflow
# ==============================================
best_params = study.best_trial.params
best_model = XGBRegressor(**best_params)
best_model.fit(X_train, y_train)

y_pred = best_model.predict(X_eval)

mae = mean_absolute_error(y_eval, y_pred)
rmse = np.sqrt(mean_squared_error(y_eval, y_pred))
r2 = r2_score(y_eval, y_pred)

print("Final tuned model performance:")
print("MAE:", mae)
print("RMSE:", rmse)
print("R²:", r2)

# Log final model
with mlflow.start_run(run_name="best_xgboost_model"):
    mlflow.log_params(best_params)
    mlflow.log_metrics({"rmse": rmse, "mae": mae, "r2": r2})
    mlflow.xgboost.log_model(best_model, name="model")

Final tuned model performance:
MAE: 32959.175436245816
RMSE: 72139.04741343326
R²: 0.9597838180022616


e:\Regression_ML_EndtoEnd\.venv\Lib\site-packages\xgboost\sklearn.py:1028: UserWarning: [22:55:35] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\c_api\c_api.cc:1427: Saving model in the UBJSON format as default.  You can use file extension: `json`, `ubj` or `deprecated` to choose between formats.
  self.get_booster().save_model(fname)
2026/03/24 22:55:43 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
